# Alpaca PAPER account (read-only)

Shows your **paper** trading account and saves your holdings for the dashboard alert.

1. **Account summary**: equity, cash, buying power
2. **Positions**: symbol, quantity, average entry, market value, unrealized P/L
3. **Recent orders**
4. **Save**: `my_positions.csv` (Symbol,Shares; the top alert in the app and `holdings_alert.py` use it) and a snapshot in
   `Reports/paper_portfolio_snapshot.csv` + `Reports/paper_account_history.csv`

**Safety:** only `https://paper-api.alpaca.markets/v2` is accepted (anything else is refused), only read-only GET requests are
made, and nothing in this notebook or in `alpaca_paper.py` can place, change or cancel an order. Keys are never printed.

**Setup (once):** add these two lines to `.env` in the project folder (values from Alpaca → Paper Trading → API keys):
```
ALPACA_PAPER_KEY_ID=...
ALPACA_PAPER_SECRET_KEY=...
```
Same from the command line: `python alpaca_paper.py` (print) or `python alpaca_paper.py --sync` (print + save);
`python run_all.py --sync-paper` refreshes `my_positions.csv` as part of the daily run.

### 1. Setup
Connect to the paper account (keys from `.env`). Set `WRITE_FILES = False` to only look, without saving files.

In [ ]:
import pandas as pd

import alpaca_paper

WRITE_FILES = True   # step 5 writes my_positions.csv and the Reports snapshot files
try:
    acct = alpaca_paper.PaperAccount()          # refuses any URL other than the paper endpoint
    print("Connected to", acct.base_url)
except alpaca_paper.PaperAccountError as e:
    acct = None
    print("Not connected:", e)
    print("Add ALPACA_PAPER_KEY_ID and ALPACA_PAPER_SECRET_KEY to .env, then run the notebook again.")

### 2. Account summary

In [ ]:
if acct:
    summary = acct.account_summary()
    display(pd.Series(summary, name="Paper account").to_frame())

### 3. Positions

In [ ]:
if acct:
    positions = acct.positions()
    display(positions.round(2))
    print(f"{len(positions)} positions, market value ${positions['Market value'].sum():,.2f}, "
          f"unrealized P/L ${positions['Unrealized P/L'].sum():,.2f}")

### 4. Recent orders (newest first, times in US Central)

In [ ]:
if acct:
    display(acct.recent_orders(limit=20))

### 5. Save `my_positions.csv` and the portfolio snapshot
The dashboard alert then compares **your** holdings with the strategy.

In [ ]:
if acct and WRITE_FILES:
    result = alpaca_paper.sync_paper_account(acct, positions_csv=alpaca_paper.POSITIONS_CSV,
                                             snapshot_csv=alpaca_paper.SNAPSHOT_CSV, history_csv=alpaca_paper.HISTORY_CSV)
    print(f"Saved {result['Positions']} positions to {result['positions_csv']}")
    print("Snapshot:", alpaca_paper.SNAPSHOT_CSV, "| history:", alpaca_paper.HISTORY_CSV)
elif acct:
    print("WRITE_FILES is False - nothing saved.")